In [0]:
storage_account = "stdataengdevrazi01"
storage_key = "gh/fNSxGWb7Ap18fL8XDOYA93goBm0I1gKrFHow8KjUllgGJfdSLl9Kv7w8uKKLVvunnW1/UWgbC+AStyOvmUA=="

# Container is 'data', folder is 'bronze'
bronze_path = f"abfss://data@{storage_account}.dfs.core.windows.net/bronze/dbo.Customers.txt"

df_bronze = spark.read \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

display(df_bronze)

In [0]:
from pyspark.sql.functions import current_timestamp, col, trim

df_silver = df_bronze \
    .dropDuplicates(["CustomerName", "City", "RegistrationDate"]) \
    .filter(col("CustomerID").isNotNull()) \
    .withColumn("CustomerName", trim(col("CustomerName"))) \
    .withColumn("City", trim(col("City"))) \
    .withColumn("IngestedAt", current_timestamp())

display(df_silver)

In [0]:
# Container is 'data', destination folder is 'silver'
silver_path = f"abfss://data@{storage_account}.dfs.core.windows.net/silver/customers_silver"

df_silver.write \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Bronze to Silver transformation completed successfully!")

In [0]:
# Read directly from the newly created Silver Delta path
df_silver_read = spark.read \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .format("delta") \
    .load(f"abfss://data@{storage_account}.dfs.core.windows.net/silver/customers_silver")

display(df_silver_read)